2026-05-27 Working

In [4]:
import os 
from pathlib import Path
root = os.getcwd()
folders = {
    "systematic_review": f"{root}/systematic_review",
        "protocol": f"{root}/systematic_review/protocol",
            "prospero": f"{root}/systematic_review/protocol/prospero",
            "cochrane": f"{root}/systematic_review/protocol/cochrane",
        "search_strategy": f"{root}/systematic_review/search_strategy",
        "search": f"{root}/systematic_review/search",
        "deduplication": f"{root}/systematic_review/deduplication",
        "screening": f"{root}/systematic_review/screening",
            "title_abstract": f"{root}/systematic_review/screening/title_abstract_screening", 
            "pdf": f"{root}/systematic_review/screening/PDF",
            "full_text": f"{root}/systematic_review/screening/full_text_screening", 
    "data_collection": f"{root}/data_collection",
        "database": f"{root}/data_collection/database",
    "meta-analysis": f"{root}/meta-analysis",
    "manuscript": f"{root}/manuscript"
}

for x, y in folders.items():
    filename = f"{x}"
    path = Path(f"{y}")
    os.makedirs(path, exist_ok = True)
    globals()[filename] = path

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from Bio import Entrez, Medline

entries = []

term_input = widgets.Text(
    placeholder="Search term",
    layout = widgets.Layout(width="75%")
)

field_tag = widgets.Dropdown(
    options=[
        ("MeSH term", "mh"),
        ("Title", "ti"),
        ("Title / Abstract", "tiab"),
        ("Publication Type", "pt"),
    ],
    value="mh",
    layout=widgets.Layout(width="15%")
)

boolean = widgets.Dropdown(
    options=["OR", "AND", "NOT", ""],
    value="",
    layout=widgets.Layout(width="10%")
)

filename_input = widgets.Text(
    placeholder="File name",
    layout=widgets.Layout(width="99.5%")
)

add_button = widgets.Button(description="Add", icon="plus", button_style="success")
delete_button = widgets.Button(description="Delete", icon="minus", button_style="danger")
clear_button = widgets.Button(description="Clear",icon="left-arrow", button_style="warning")
save_button = widgets.Button(description="Save", icon="save", button_style="info")

output = widgets.Output()

def build_query(entries):
    parts = []
    current_or_group = []

    for entry in entries:
        term = entry["term"].strip()
        field = entry["field"]
        op = entry["boolean"]

        if not term:
            continue

        current_or_group.append(f'"{term}"[{field}]')

        if op == "OR":
            continue

        parts.append("(" + " OR ".join(current_or_group) + ")")
        current_or_group = []

        if op in ("AND", "NOT"):
            parts.append(op)

    if current_or_group:
        parts.append("(" + " OR ".join(current_or_group) + ")")

    return " ".join(parts)


def refresh_output(message=""):
    with output:
        clear_output()
        if message:
            print(message)
            print()

        print("Entries:")
        if entries:
            for i, entry in enumerate(entries, start=1):
                op_label = entry["boolean"] if entry["boolean"] != "" else "END"
                print(f'{i}. "{entry["term"]}" [{entry["field"]}] -> {op_label}')
        else:
            print("[none]")

        print("\nCurrent query:")
        query = build_query(entries)
        print(query if query else "[empty]")


def add_entry(_):
    term = term_input.value.strip()
    field = field_tag.value
    op = boolean.value

    if not term:
        refresh_output("Please enter a term.")
        return

    entries.append({
        "term": term,
        "field": field,
        "boolean": op
    })

    term_input.value = ""
    refresh_output(f'Added: "{term}"[{field}] -> {op if op else "END"}')


def delete_last_entry(_):
    if not entries:
        refresh_output("Nothing to delete.")
        return

    removed = entries.pop()
    refresh_output(
        f'Removed: "{removed["term"]}"[{removed["field"]}] -> {removed["boolean"] if removed["boolean"] else "END"}'
    )


def clear_all_entries(_):
    entries.clear()
    refresh_output("Cleared all entries.")


def save_query(_):
    query = build_query(entries)
    filename = filename_input.value.strip() or "default_strategy"
    filepath = f"{search_strategy}/{filename}.txt"

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(query)

    refresh_output(f"Saved query to {filepath}")
    return filepath

add_button.on_click(add_entry)
delete_button.on_click(delete_last_entry)
clear_button.on_click(clear_all_entries)
save_button.on_click(save_query)

entry_row = widgets.HBox(children = [term_input, field_tag, boolean], layout = {"width":"100%"})
buttons = widgets.HBox(children = [add_button, delete_button, clear_button, save_button], layout = {"width":"100%"})
controls = widgets.VBox(children = [filename_input, entry_row, buttons, output], layout = {"width":"100%"})

display(controls)

In [3]:
filename = filename_input.value
import os
with open(f"./data/{filename}.txt", "r") as f:
    query = f.read()
# use NCBI's e-utitilies to pull PMIDs using e-search.

query = f"{query}"
Entrez.email = "dkim246@jhmi.edu"
Entrez.api_key = 'bb1c481d8e167acd16f3616593c18b3aab08'

handle = Entrez.esearch(db= "pubmed", 
                        term = query, 
                        usehistory = "y", 
                        retmax = 2000,
                        retmode = "xml")

pmid = Entrez.read(handle)

pmid = pmid['IdList']
pmid = ",".join(pmid) # list to string
#with open(f"./data/pmid_{filename}.txt", 'w') as f:
#    f.write(pmid)
os.makedirs(f"./data/pubmed/pmid/", exist_ok = True)
with open(f"./data/pubmed/pmid/{filename}.txt", 'w') as f:
    f.write(pmid)
handle.close()

# ncbi e-summary
handle = Entrez.esummary(db= "pubmed", 
                         id = pmid, 
                         retmode = "xml", 
                         usehistory = "y", 
                         retmax = 2000)

xml = handle.read()
#xml_file = f"./data/{filename}.xml"
os.makedirs(f"./data/pubmed/xml/", exist_ok = True)
xml_file = f"./data/pubmed/xml/{filename}.xml"
with open(xml_file, "wb") as f:
    f.write(xml)   
handle.close()

import xml.etree.ElementTree as ET

tree = ET.parse(f"{xml_file}")
root = tree.getroot()

docsum = root[0]

def xml_parse(docsum):
    df = {}
    df["pmid"] = docsum.find("Id").text
    for item in docsum.findall("Item"):
        key = item.attrib.get("Name")
        if item.attrib.get("Type") == "List":
            values = [sub.text for sub in item.findall("Item") if sub.text]
            df[key] = values
        else:
            df[key] = item.text
    return df
records = [xml_parse(doc) for doc in root.findall(".//DocSum")]
df = pd.DataFrame(records)

os.makedirs(f"./data/pubmed/", exist_ok = True)
csv_file = f"./data/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")

# using e-fetch, the abstracts are pulled

handle = Entrez.efetch(
    db="pubmed",
    id=pmid,
    rettype="medline",
    retmode="text"
)

text = list(Medline.parse(handle))
data = pd.DataFrame(text)
data_csv = data.map(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)
os.makedirs(f"./data/pubmed/medline/", exist_ok = True)
data_csv.to_csv(f"./data/pubmed/medline/{filename.replace("pm","md")}.csv", index=False)
globals()[f"{filename.replace("pm","md")}"] = data
handle.close()

abstracts = pd.DataFrame(text)[["PMID", "AB"]]
abstracts.rename(columns = {"PMID":"pmid", "AB":"abstract"}, inplace = True)
df = df.merge(abstracts, on = "pmid", how = "left")
df['year'] = df['PubDate'].str[:4]
csv_file = f"./data/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")
num = len(df)
print(f"Number of records found: {num}")
data.info()



FileNotFoundError: [Errno 2] No such file or directory: './data/.txt'